# 06 - Camada Gold: Modelagem Dimensional, KPIs e Destino Duplo (Modo Append)
**Squad 2 — Real Time for Business | Unificação Dupla 1 & Dupla 5**  
**Integrantes:** Zaiden Emiliano Segundo Seleme & Lucas Sousa Santos Oliveira  
**Data Marts de Escopo:**
1. `gold_dim_produtos`
2. `gold_metricas_categorias`
3. `gold_logistica_pedidos_rota`
4. `gold_performance_transportadoras`
5. `gold_distribuicao_geografica_clientes`

**Branch:** `feat/squad2-dupla1e5`

---

### Diretrizes Arquiteturais da Camada Gold:
1. **Consumo Exclusivo Silver-to-Gold:** Leitura restrita aos dados curados da Camada Silver em Delta Lake. Zero leitura de raw/bronze.
2. **Proteção contra ausência de tabelas Silver:** Se uma tabela Silver estiver ausente ou vazia, retorna DataFrame vazio tipado com aviso explícito.
3. **Modelagem Dimensional e KPIs:**
   * **Dimensão de Produtos:** Desnormalização com hierarquia pai/filha de categorias, segmentação por faixas de preço e flags de disponibilidade.
   * **Métricas de Categorias:** Agregações de catálogo, taxa de disponibilidade, ticket médio e dispersão.
   * **Logística de Entregas:** Throughput diário de saídas para rota, entregas concluídas, falhas operacionais e taxa de sucesso percentual.
   * **Scorecard de Transportadoras:** Ranking de eficiência operacional por data de evento via `dense_rank()`.
   * **Inteligência Geográfica:** Distribuição e densidade de clientes e endereços cadastrados por estado (UF).
4. **Destino Duplo Mandatório (*Dual Sink*) em Modo APPEND:**
   * Persistência em **modo `append`** com carimbo temporal (`gold_processed_at`), garantindo a historização de snapshots analíticos e evolução de KPIs.
   * **Proteção contra persistência vazia:** Data Marts vazios são ignorados com aviso, evitando gravar lotes sem dados.
   * **ADLS Gen2:** Tabelas analíticas em Delta Lake com modo `append`, ACID e compactação automática (`autoCompact`, `optimizeWrite`).
   * **Azure SQL Server:** Replicação relacional acumulada no schema `squad2` (modo `append`).
5. **Auditoria de Fechamento Matemático:** Validação estrita de integridade para apresentação ao PO.

In [ ]:
# Instalação de dependências no escopo da sessão Serverless
%pip install python-dotenv==1.0.1 azure-storage-file-datalake azure-identity pandas pyodbc --quiet

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os
import pandas as pd

# Padronização de TimeZone Oficial da Squad 2 (Horário de Brasília - UTC-3)
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
from dotenv import load_dotenv, find_dotenv
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, BooleanType, TimestampType
)

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env")
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

# Credenciais ADLS Gen2
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

# Credenciais SQL Server
sql_host = os.getenv("SQL_HOST", "srv-database-intership.database.windows.net")
sql_database = os.getenv("SQL_DATABASE", "internshipDatabase")
sql_user = os.getenv("SQL_USERNAME", "estagiario_user")
sql_password = os.getenv("SQL_PASSWORD")

assert client_id and client_secret and sql_password, "Credenciais ADLS/SQL incompletas no .env"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

jdbc_url = f"jdbc:sqlserver://{sql_host}:1433;database={sql_database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

base_squad = f"abfss://squad2@{storage_account}.dfs.core.windows.net"

caminhos = {
    "silver_produtos": f"{base_squad}/silver/ecommerce_produtos",
    "silver_categorias": f"{base_squad}/silver/ecommerce_categorias",
    "silver_rastreamento": f"{base_squad}/silver/ecommerce_rastreamento_entregas",
    "silver_enderecos": f"{base_squad}/silver/ecommerce_enderecos",
    "silver_pedidos": f"{base_squad}/silver/ecommerce_pedidos",

    "gold_dim_produtos": f"{base_squad}/gold/gold_dim_produtos",
    "gold_metricas_categorias": f"{base_squad}/gold/gold_metricas_categorias",
    "gold_pedidos_rota": f"{base_squad}/gold/gold_logistica_pedidos_rota",
    "gold_performance_transportadoras": f"{base_squad}/gold/gold_performance_transportadoras",
    "gold_distribuicao_geografica": f"{base_squad}/gold/gold_distribuicao_geografica_clientes",
    "gold_alertas_operacionais": f"{base_squad}/gold/gold_alertas_operacionais"
}

def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

# Schemas de contingência para leitura defensiva
schema_spark_produtos = StructType([
    StructField("sku", StringType(), True),
    StructField("nome_produto", StringType(), True),
    StructField("descricao", StringType(), True),
    StructField("id_categoria", LongType(), True),
    StructField("preco_lista", DoubleType(), True),
    StructField("unidade_medida", StringType(), True),
    StructField("nome_marca", StringType(), True),
    StructField("is_ativo", BooleanType(), True),
    StructField("bronze_source_file", StringType(), True),
    StructField("bronze_ingested_at", TimestampType(), True),
    StructField("silver_processed_at", TimestampType(), True)
])

schema_spark_categorias = StructType([
    StructField("id_categoria", LongType(), True),
    StructField("nome_categoria", StringType(), True),
    StructField("id_categoria_pai", LongType(), True),
    StructField("nome_categoria_pai", StringType(), True),
    StructField("tipo_categoria", StringType(), True)
])

schema_spark_rastreamento = StructType([
    StructField("id_rastreamento", LongType(), True),
    StructField("id_pedido_ecommerce", LongType(), True),
    StructField("codigo_rastreio", StringType(), True),
    StructField("id_transportadora", LongType(), True),
    StructField("nome_transportadora", StringType(), True),
    StructField("status_entrega", StringType(), True),
    StructField("dt_evento", TimestampType(), True),
    StructField("observacao", StringType(), True)
])

schema_spark_enderecos = StructType([
    StructField("id_endereco", LongType(), True),
    StructField("id_cliente", LongType(), True),
    StructField("logradouro", StringType(), True),
    StructField("numero", LongType(), True),
    StructField("complemento", StringType(), True),
    StructField("bairro", StringType(), True),
    StructField("cep", StringType(), True),
    StructField("cidade", StringType(), True),
    StructField("estado", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("apelido", StringType(), True),
    StructField("is_principal", BooleanType(), True)
])

schema_spark_pedidos = StructType([
    StructField("id_pedido", LongType(), True),
    StructField("dt_pedido", TimestampType(), True)
])

print("Ambiente e caminhos da Camada Gold preparados com proteção de leitura!")


## 1. Leitura das Tabelas Curadas da Camada Silver (Delta Lake)

In [ ]:
def carregar_silver_seguro(caminho, schema_spark, nome):
    if not tabela_delta_existe(caminho):
        print(f"  [AVISO] Tabela Silver '{nome}' não encontrada. Retornando DataFrame vazio para continuidade da esteira.")
        return spark.createDataFrame([], schema_spark)
    return spark.read.format("delta").options(**adls_options).load(caminho)

print("Lendo tabelas curadas da Camada Silver de forma defensiva...")
df_silver_produtos = carregar_silver_seguro(caminhos["silver_produtos"], schema_spark_produtos, "ecommerce_produtos")
df_silver_categorias = carregar_silver_seguro(caminhos["silver_categorias"], schema_spark_categorias, "ecommerce_categorias")
df_silver_rastreamento = carregar_silver_seguro(caminhos["silver_rastreamento"], schema_spark_rastreamento, "ecommerce_rastreamento_entregas")
df_silver_enderecos = carregar_silver_seguro(caminhos["silver_enderecos"], schema_spark_enderecos, "ecommerce_enderecos")
df_silver_pedidos = carregar_silver_seguro(caminhos["silver_pedidos"], schema_spark_pedidos, "ecommerce_pedidos")

print(f"  Produtos:     {df_silver_produtos.count()} registros carregados.")
print(f"  Categorias:   {df_silver_categorias.count()} registros carregados.")
print(f"  Rastreamento: {df_silver_rastreamento.count()} registros carregados.")
print(f"  Endereços:    {df_silver_enderecos.count()} registros carregados.")
if df_silver_rastreamento.count() > 0 and "status_entrega" in df_silver_rastreamento.columns:
    print("  -> Distribuição de Status em Rastreamento (Silver):")
    for r_st in df_silver_rastreamento.groupBy("status_entrega").count().collect():
        print(f"     * {r_st['status_entrega']}: {r_st['count']} eventos")


## 2. Construção dos Data Marts de Produtos e Categorias

In [ ]:
# 1. Data Mart 1: gold_dim_produtos
df_cat_lookup = df_silver_categorias.select(
    F.col("id_categoria").cast(StringType()),
    F.col("nome_categoria"),
    F.col("id_categoria_pai").cast(StringType()),
    F.col("nome_categoria_pai"),
    F.col("tipo_categoria")
)

df_gold_dim_produtos = (
    df_silver_produtos
    .withColumn("id_categoria", F.col("id_categoria").cast(StringType()))
    .join(F.broadcast(df_cat_lookup), on="id_categoria", how="left")
    .withColumn("preco_lista", F.col("preco_lista").cast(DoubleType()))
    .withColumn("preco_custo", F.col("preco_custo").cast(DoubleType()) if "preco_custo" in df_silver_produtos.columns else F.lit(None).cast(DoubleType()))
    .withColumn("margem_bruta_unitaria",
        F.when(F.col("preco_custo").isNotNull(), F.round(F.col("preco_lista") - F.col("preco_custo"), 2))
         .otherwise(F.lit(None).cast(DoubleType()))
    )
    .withColumn("margem_lucro_pct",
        F.when(F.col("preco_custo").isNotNull() & (F.col("preco_lista") > 0),
               F.round(((F.col("preco_lista") - F.col("preco_custo")) / F.col("preco_lista")) * 100, 2))
         .otherwise(F.lit(None).cast(DoubleType()))
    )
    .withColumn("status_rentabilidade",
        F.when(F.col("preco_custo").isNull(), F.lit("Sem Custo Cadastrado"))
         .when(F.col("preco_lista") < F.col("preco_custo"), F.lit("Margem Negativa"))
         .when(F.col("margem_lucro_pct") >= 40.0, F.lit("Alta Margem (>=40%)"))
         .otherwise(F.lit("Margem Saudavel"))
    )
    .withColumn("status_disponibilidade", F.when(F.col("is_ativo") == True, F.lit("Ativo")).otherwise(F.lit("Inativo")))
    .withColumn(
        "faixa_preco",
        F.when(F.col("preco_lista") < 50, F.lit("1. Economica (< R$ 50)"))
        .when((F.col("preco_lista") >= 50) & (F.col("preco_lista") < 200), F.lit("2. Padrao (R$ 50 - 200)"))
        .when((F.col("preco_lista") >= 200) & (F.col("preco_lista") < 1000), F.lit("3. Premium (R$ 200 - 1000)"))
        .otherwise(F.lit("4. Luxo (>= R$ 1000)"))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .select(
        "sku", "nome_produto", "descricao", "id_categoria", "nome_categoria",
        "id_categoria_pai", "nome_categoria_pai", "tipo_categoria", "nome_marca",
        "unidade_medida", "preco_lista", "preco_custo", "margem_bruta_unitaria",
        "margem_lucro_pct", "status_rentabilidade", "is_ativo", "status_disponibilidade",
        "faixa_preco", "gold_processed_at"
    )
)

print(f"Data Mart 1 (gold_dim_produtos): {df_gold_dim_produtos.count()} produtos gerados.")
if df_gold_dim_produtos.count() > 0:
    display(df_gold_dim_produtos.limit(5))

# 2. Data Mart 2: gold_metricas_categorias
df_gold_metricas_categorias = (
    df_gold_dim_produtos
    .groupBy(
        "id_categoria",
        "nome_categoria",
        "id_categoria_pai",
        "nome_categoria_pai",
        "tipo_categoria"
    )
    .agg(
        F.count("sku").alias("total_skus_cadastrados"),
        F.sum(F.when(F.col("is_ativo") == True, 1).otherwise(0)).alias("total_skus_ativos"),
        F.sum(F.when(F.col("is_ativo") == False, 1).otherwise(0)).alias("total_skus_inativos"),
        F.round(F.avg("preco_lista"), 2).alias("preco_medio_lista"),
        F.round(F.min("preco_lista"), 2).alias("preco_minimo_lista"),
        F.round(F.max("preco_lista"), 2).alias("preco_maximo_lista"),
        F.round(F.max("preco_lista") - F.min("preco_lista"), 2).alias("amplitude_preco"),
        F.countDistinct("nome_marca").alias("total_marcas_distintas")
    )
    .withColumn(
        "taxa_disponibilidade_pct",
        F.when(F.col("total_skus_cadastrados") > 0, F.round((F.col("total_skus_ativos") / F.col("total_skus_cadastrados")) * 100, 2)).otherwise(F.lit(0.0))
    )
    .withColumn(
        "is_categoria_raiz",
        F.when(F.col("id_categoria_pai").isNull(), F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn("gold_calculated_at", F.current_timestamp())
    .select(
        "id_categoria", "nome_categoria", "id_categoria_pai", "nome_categoria_pai",
        "tipo_categoria", "is_categoria_raiz", "total_skus_cadastrados", "total_skus_ativos",
        "total_skus_inativos", "taxa_disponibilidade_pct", "preco_medio_lista", "preco_minimo_lista",
        "preco_maximo_lista", "amplitude_preco", "total_marcas_distintas", "gold_calculated_at"
    )
    .orderBy(F.desc("total_skus_cadastrados"))
)

print(f"Data Mart 2 (gold_metricas_categorias): {df_gold_metricas_categorias.count()} categorias geradas.")
if df_gold_metricas_categorias.count() > 0:
    display(df_gold_metricas_categorias.limit(5))

## 3. Construção dos Data Marts de Logística e Endereços

In [ ]:
# 3. Data Mart 3: gold_logistica_pedidos_rota (Throughput e KPI Negócio 7: SLA de Entrega <= 7 dias)
# Enriquecimento com dt_pedido para aferição do SLA de 7 dias
if "id_pedido" in df_silver_pedidos.columns and "dt_pedido" in df_silver_pedidos.columns and df_silver_pedidos.count() > 0:
    df_ped_lookup = (
        df_silver_pedidos
        .select(
            F.col("id_pedido").cast(LongType()).alias("fk_pedido_id"),
            F.col("dt_pedido")
        )
        .distinct()
    )
    df_rast_com_pedido = df_silver_rastreamento.join(
        F.broadcast(df_ped_lookup),
        df_silver_rastreamento["id_pedido_ecommerce"] == df_ped_lookup["fk_pedido_id"],
        "left"
    )
else:
    df_rast_com_pedido = df_silver_rastreamento.withColumn("dt_pedido", F.lit(None).cast(TimestampType()))

df_gold_pedidos_rota = (
    df_rast_com_pedido
    .withColumn("status_norm", F.translate(F.regexp_replace(F.lower(F.trim(F.col("status_entrega"))), r"\s+", "_"), "çãáéêíóôú", "caaeeioou"))
    .withColumn("data_evento", F.to_date("dt_evento"))
    .withColumn(
        "dias_entrega",
        F.when(
            (F.col("status_norm") == "entregue") & F.col("dt_pedido").isNotNull(),
            F.datediff(F.col("dt_evento"), F.col("dt_pedido"))
        ).otherwise(F.lit(None))
    )
    .groupBy("data_evento")
    .agg(
        F.count("id_rastreamento").alias("total_eventos"),
        F.countDistinct("id_pedido_ecommerce").alias("total_pedidos_unicos"),
        # Contagem abrangente de TODOS os status operacionais
        F.sum(F.when(F.col("status_norm") == "em_separacao", 1).otherwise(0)).alias("total_em_separacao"),
        F.sum(F.when(F.col("status_norm") == "coletado", 1).otherwise(0)).alias("total_coletados"),
        F.sum(F.when(F.col("status_norm") == "em_transito", 1).otherwise(0)).alias("total_em_transito"),
        F.sum(F.when(F.col("status_norm") == "saiu_para_entrega", 1).otherwise(0)).alias("total_em_rota"),
        # KPI Negócio 6: Throughput última milha em tempo real (últimas 2 horas)
        F.sum(
            F.when(
                (F.col("status_norm") == "saiu_para_entrega") &
                (F.col("dt_evento") >= (F.current_timestamp() - F.expr("INTERVAL 2 HOURS"))),
                1
            ).otherwise(0)
        ).alias("pedidos_em_rota_ultimas_2h"),
        F.sum(F.when(F.col("status_norm") == "entregue", 1).otherwise(0)).alias("total_entregues"),
        F.sum(F.when(F.col("status_norm").isin("falha_na_entrega", "devolvido"), 1).otherwise(0)).alias("total_falhas"),
        F.sum(F.when(F.col("status_norm") == "cancelado", 1).otherwise(0)).alias("total_cancelados"),
        F.sum(F.when(~F.col("status_norm").isin("em_separacao", "coletado", "em_transito", "saiu_para_entrega", "entregue", "falha_na_entrega", "devolvido", "cancelado"), 1).otherwise(0)).alias("total_outros_status"),
        # Métricas de SLA (Regra Negócio 7: SLA = 7 dias)
        F.sum(F.when((F.col("status_norm") == "entregue") & (F.col("dias_entrega") <= 7), 1).otherwise(0)).alias("total_entregues_no_prazo"),
        F.sum(F.when((F.col("status_norm") == "entregue") & (F.col("dias_entrega") > 7), 1).otherwise(0)).alias("total_entregues_com_atraso")
    )
    .withColumn(
        "taxa_sucesso_pct",
        F.when(
            (F.col("total_entregues") + F.col("total_falhas")) > 0,
            F.round((F.col("total_entregues") / (F.col("total_entregues") + F.col("total_falhas"))) * 100, 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "taxa_sla_entrega_pct",
        F.when(
            F.col("total_entregues") > 0,
            F.round((F.col("total_entregues_no_prazo") / F.col("total_entregues")) * 100, 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .select(
        "data_evento", "total_eventos", "total_pedidos_unicos",
        "total_em_separacao", "total_coletados", "total_em_transito", "total_em_rota",
        "pedidos_em_rota_ultimas_2h", "total_entregues", "total_falhas",
        "total_cancelados", "total_outros_status",
        "total_entregues_no_prazo", "total_entregues_com_atraso", "taxa_sla_entrega_pct",
        "taxa_sucesso_pct", "gold_processed_at"
    )
    .orderBy(F.desc("data_evento"))
)

print(f"Data Mart 3 (gold_logistica_pedidos_rota): {df_gold_pedidos_rota.count()} datas geradas.")
if df_gold_pedidos_rota.count() > 0:
    display(df_gold_pedidos_rota.limit(5))

# 4. Data Mart 4: gold_performance_transportadoras
w_rank_transportadora = Window.partitionBy("data_evento").orderBy(F.desc("volume_movimentado"))

df_gold_transportadoras = (
    df_silver_rastreamento
    .withColumn("status_norm", F.translate(F.regexp_replace(F.lower(F.trim(F.col("status_entrega"))), r"\s+", "_"), "çãáéêíóôú", "caaeeioou"))
    .withColumn("data_evento", F.to_date("dt_evento"))
    .groupBy("data_evento", "id_transportadora")
    .agg(
        F.count("id_rastreamento").alias("volume_movimentado"),
        F.countDistinct("id_pedido_ecommerce").alias("total_pedidos_atendidos"),
        F.sum(F.when(F.col("status_norm") == "entregue", 1).otherwise(0)).alias("volume_entregue"),
        F.sum(F.when(F.col("status_norm").isin("falha_na_entrega", "devolvido"), 1).otherwise(0)).alias("volume_falhas")
    )
    .withColumn(
        "taxa_eficiencia_pct",
        F.when(
            (F.col("volume_entregue") + F.col("volume_falhas")) > 0,
            F.round((F.col("volume_entregue") / (F.col("volume_entregue") + F.col("volume_falhas"))) * 100, 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("ranking_volume_dia", F.dense_rank().over(w_rank_transportadora).cast("int"))
    .withColumn("gold_processed_at", F.current_timestamp())
    .select(
        "data_evento", "id_transportadora", "volume_movimentado", "total_pedidos_atendidos",
        "volume_entregue", "volume_falhas", "taxa_eficiencia_pct", "ranking_volume_dia",
        "gold_processed_at"
    )
    .orderBy("data_evento", "ranking_volume_dia")
)

print(f"Data Mart 4 (gold_performance_transportadoras): {df_gold_transportadoras.count()} linhas de ranking geradas.")
if df_gold_transportadoras.count() > 0:
    display(df_gold_transportadoras.limit(5))

# 5. Data Mart 5: gold_distribuicao_geografica_clientes
total_clientes_base = df_silver_enderecos.select("id_cliente").distinct().count()

df_gold_geografia = (
    df_silver_enderecos
    .groupBy("estado")
    .agg(
        F.count("id_endereco").alias("total_enderecos"),
        F.countDistinct("id_cliente").alias("total_clientes_unicos"),
        F.sum(F.when(F.col("is_principal") == True, 1).otherwise(0)).alias("total_enderecos_principais")
    )
    .withColumn(
        "penetracao_clientes_pct",
        F.when(F.lit(total_clientes_base) > 0, F.round((F.col("total_clientes_unicos") / F.lit(total_clientes_base)) * 100, 2)).otherwise(F.lit(0.0))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .select(
        "estado", "total_enderecos", "total_clientes_unicos", "total_enderecos_principais",
        "penetracao_clientes_pct", "gold_processed_at"
    )
    .orderBy(F.desc("total_clientes_unicos"))
)

print(f"Data Mart 5 (gold_distribuicao_geografica_clientes): {df_gold_geografia.count()} UFs geradas.")
if df_gold_geografia.count() > 0:
    display(df_gold_geografia.limit(5))

# 6. Data Mart 6: gold_alertas_operacionais (Alertas de Negócio em Tempo Real)
schema_alertas = StructType([
    StructField("data_alerta", TimestampType(), True),
    StructField("dominio", StringType(), True),
    StructField("regra_negocio", StringType(), True),
    StructField("chave_entidade", StringType(), True),
    StructField("valor_detectado", StringType(), True),
    StructField("severidade", StringType(), True),
    StructField("gold_processed_at", TimestampType(), True)
])

lista_alertas_dfs = []

# Alerta 1: Endereços Excessivos por Cliente (Regra Negócio 5 de ecommerce_enderecos)
if df_silver_enderecos.count() > 0:
    al_end = (
        df_silver_enderecos
        .groupBy("id_cliente")
        .agg(F.count("id_endereco").alias("qtd_end"))
        .filter(F.col("qtd_end") > 3)
        .select(
            F.current_timestamp().alias("data_alerta"),
            F.lit("ecommerce_enderecos").alias("dominio"),
            F.lit("Negócio 5: Endereços Excessivos por Cliente").alias("regra_negocio"),
            F.concat(F.lit("id_cliente: "), F.col("id_cliente").cast(StringType())).alias("chave_entidade"),
            F.concat(F.col("qtd_end").cast(StringType()), F.lit(" endereços cadastrados (limite: 3)")).alias("valor_detectado"),
            F.lit("ALTA").alias("severidade"),
            F.current_timestamp().alias("gold_processed_at")
        )
    )
    if al_end.count() > 0:
        lista_alertas_dfs.append(al_end)

# Alerta 2: Preço Inválido em Produto Ativo (Regra Negócio 5 de ecommerce_produtos)
if df_silver_produtos.count() > 0:
    al_prod = (
        df_silver_produtos
        .filter((F.col("is_ativo") == True) & (F.col("preco_lista") <= 0))
        .select(
            F.current_timestamp().alias("data_alerta"),
            F.lit("ecommerce_produtos").alias("dominio"),
            F.lit("Negócio 5: Produto Ativo com Preço Zero/Negativo").alias("regra_negocio"),
            F.concat(F.lit("sku: "), F.col("sku")).alias("chave_entidade"),
            F.concat(F.lit("preco_lista: R$ "), F.col("preco_lista").cast(StringType())).alias("valor_detectado"),
            F.lit("CRÍTICA").alias("severidade"),
            F.current_timestamp().alias("gold_processed_at")
        )
    )
    if al_prod.count() > 0:
        lista_alertas_dfs.append(al_prod)

# Alerta 3: Entrega Duplicada para o Mesmo Pedido (Regra Negócio 8 de ecommerce_rastreamento_entregas)
if df_silver_rastreamento.count() > 0:
    al_log = (
        df_silver_rastreamento
        .filter(F.col("status_entrega") == "entregue")
        .groupBy("id_pedido_ecommerce")
        .agg(F.count("id_rastreamento").alias("qtd_entregas"))
        .filter(F.col("qtd_entregas") > 1)
        .select(
            F.current_timestamp().alias("data_alerta"),
            F.lit("ecommerce_rastreamento_entregas").alias("dominio"),
            F.lit("Negócio 8: Evento 'entregue' Duplicado").alias("regra_negocio"),
            F.concat(F.lit("id_pedido: "), F.col("id_pedido_ecommerce").cast(StringType())).alias("chave_entidade"),
            F.concat(F.col("qtd_entregas").cast(StringType()), F.lit(" confirmações de entrega recebidas")).alias("valor_detectado"),
            F.lit("ALTA").alias("severidade"),
            F.current_timestamp().alias("gold_processed_at")
        )
    )
    if al_log.count() > 0:
        lista_alertas_dfs.append(al_log)

# Alerta 4: Encomenda Sumida / Sem Rastreio > 3 dias a partir da coleta (Regra Negócio 10 de ecommerce_rastreamento_entregas)
# Contempla todo o fluxo a partir de 'coletado': coletado, em_transito, saiu_para_entrega
if df_silver_rastreamento.count() > 0:
    w_ped = Window.partitionBy("id_pedido_ecommerce").orderBy(F.col("dt_evento").desc())
    status_pos_coleta = ["coletado", "em_transito", "saiu_para_entrega"]
    al_sumidos = (
        df_silver_rastreamento
        .withColumn("rn", F.row_number().over(w_ped))
        .filter(F.col("rn") == 1)
        .filter(F.col("status_entrega").isin(status_pos_coleta))
        .filter(F.datediff(F.current_timestamp(), F.col("dt_evento")) > 3)
        .select(
            F.current_timestamp().alias("data_alerta"),
            F.lit("ecommerce_rastreamento_entregas").alias("dominio"),
            F.concat(F.lit("Negócio 10: Encomenda Estagnada em '"), F.col("status_entrega"), F.lit("' (> 3 dias)")).alias("regra_negocio"),
            F.concat(F.lit("id_pedido: "), F.col("id_pedido_ecommerce").cast(StringType())).alias("chave_entidade"),
            F.concat(F.datediff(F.current_timestamp(), F.col("dt_evento")).cast(StringType()), F.lit(" dias sem movimentação a partir da coleta")).alias("valor_detectado"),
            F.lit("CRÍTICA").alias("severidade"),
            F.current_timestamp().alias("gold_processed_at")
        )
    )
    if al_sumidos.count() > 0:
        lista_alertas_dfs.append(al_sumidos)

# Alerta 5: Mudança Não Planejada no Total de Categorias Raiz (Regra Negócio 2 de ecommerce_categorias)
if df_silver_categorias.count() > 0:
    qtd_raizes_atual = (
        df_silver_categorias
        .filter(F.col("id_categoria_pai").isNull() | (F.col("id_categoria_pai") == 0))
        .select("id_categoria")
        .distinct()
        .count()
    )
    # Compara contra baseline histórico na Camada Gold ou execuções prévias
    if tabela_delta_existe(caminhos["gold_metricas_categorias"]):
        df_gold_cat_exist = spark.read.format("delta").options(**adls_options).load(caminhos["gold_metricas_categorias"])
        qtd_raizes_hist = (
            df_gold_cat_exist
            .filter(F.col("is_categoria_raiz") == True)
            .select("id_categoria")
            .distinct()
            .count()
        )
    else:
        qtd_raizes_hist = qtd_raizes_atual

    if qtd_raizes_hist > 0 and qtd_raizes_atual != qtd_raizes_hist:
        al_cat = spark.createDataFrame([
            (
                "ecommerce_categorias",
                "Negócio 2: Alteração no Número de Categorias Raiz",
                "total_categorias_raiz",
                f"Atual: {qtd_raizes_atual} | Anterior: {qtd_raizes_hist} (delta: {qtd_raizes_atual - qtd_raizes_hist})",
                "ALTA"
            )
        ], ["dominio", "regra_negocio", "chave_entidade", "valor_detectado", "severidade"]) \
        .withColumn("data_alerta", F.current_timestamp()) \
        .withColumn("gold_processed_at", F.current_timestamp()) \
        .select("data_alerta", "dominio", "regra_negocio", "chave_entidade", "valor_detectado", "severidade", "gold_processed_at")
        
        lista_alertas_dfs.append(al_cat)

# Alerta 6: Volume Excessivo de Novos SKUs / Suspeita de Carga de Teste (Regra Negócio 4 de ecommerce_produtos)
# Lógica Idempotente Inteligente:
# 1. Se a tabela gold_alertas_operacionais já existir, filtra lotes inéditos (anti-join) para alertar apenas o novo lote e evitar duplicações.
# 2. Se a tabela ainda não existir (primeira carga/backfill), audita e alerta todos os lotes que excederam 50 SKUs.
if df_silver_produtos.count() > 0:
    col_lote = "bronze_source_file" if "bronze_source_file" in df_silver_produtos.columns else ("silver_processed_at" if "silver_processed_at" in df_silver_produtos.columns else None)
    
    if col_lote:
        # Agrupa por lote e calcula a quantidade de SKUs distintos
        df_lotes_resumo = (
            df_silver_produtos
            .groupBy(col_lote)
            .agg(F.countDistinct("sku").alias("qtd_novos_skus"))
            .filter(F.col("qtd_novos_skus") > 50)
            .withColumn("chave_entidade", F.concat(F.lit("lote: "), F.substring_index(F.col(col_lote).cast(StringType()), "/", -2)))
        )
        
        # Anti-join contra histórico já alertado para processamento incremental
        if tabela_delta_existe(caminhos["gold_alertas_operacionais"]):
            df_al_exist = spark.read.format("delta").options(**adls_options).load(caminhos["gold_alertas_operacionais"])
            df_lotes_anormais = df_lotes_resumo.join(
                F.broadcast(df_al_exist.filter(F.col("dominio") == "ecommerce_produtos").select("chave_entidade")),
                "chave_entidade",
                "left_anti"
            )
        else:
            df_lotes_anormais = df_lotes_resumo
        
        if df_lotes_anormais.count() > 0:
            al_novos_skus = (
                df_lotes_anormais
                .select(
                    F.current_timestamp().alias("data_alerta"),
                    F.lit("ecommerce_produtos").alias("dominio"),
                    F.lit("Negócio 4: Volume Anormal de Novos SKUs (> 50 no micro-lote)").alias("regra_negocio"),
                    F.col("chave_entidade"),
                    F.concat(F.col("qtd_novos_skus").cast(StringType()), F.lit(" novos SKUs adicionados (limite: 50) - Possível carga de teste")).alias("valor_detectado"),
                    F.lit("MÉDIA").alias("severidade"),
                    F.current_timestamp().alias("gold_processed_at")
                )
            )
            lista_alertas_dfs.append(al_novos_skus)


# Alerta 7: Margem Negativa / Prejuízo Unitário (Regra Negócio 7 de ecommerce_produtos)
if "preco_custo" in df_silver_produtos.columns and df_silver_produtos.count() > 0:
    al_margem_neg = (
        df_silver_produtos
        .filter(F.col("preco_custo").isNotNull() & (F.col("preco_custo") > F.col("preco_lista")))
        .select(
            F.current_timestamp().alias("data_alerta"),
            F.lit("ecommerce_produtos").alias("dominio"),
            F.lit("Negócio 7: Margem Negativa (Preço de Custo > Preço de Lista)").alias("regra_negocio"),
            F.concat(F.lit("sku: "), F.col("sku")).alias("chave_entidade"),
            F.concat(F.lit("Custo: R$ "), F.col("preco_custo").cast(StringType()), F.lit(" | Venda: R$ "), F.col("preco_lista").cast(StringType())).alias("valor_detectado"),
            F.lit("ALTA").alias("severidade"),
            F.current_timestamp().alias("gold_processed_at")
        )
    )
    if al_margem_neg.count() > 0:
        lista_alertas_dfs.append(al_margem_neg)

if lista_alertas_dfs:
    df_gold_alertas_operacionais = lista_alertas_dfs[0]
    for df_a in lista_alertas_dfs[1:]:
        df_gold_alertas_operacionais = df_gold_alertas_operacionais.unionByName(df_a)
else:
    df_gold_alertas_operacionais = spark.createDataFrame([], schema=schema_alertas)

print(f"Data Mart 6 (gold_alertas_operacionais): {df_gold_alertas_operacionais.count()} alertas ativos detectados.")
if df_gold_alertas_operacionais.count() > 0:
    display(df_gold_alertas_operacionais.limit(10))


## 4. Persistência 1: Gravação em Delta Lake no ADLS Gen2 (Sincronização / Modo Overwrite)

**Prevenção de Inchaço e Explosão de Volumetria:**
Para evitar duplicações de dados de dimensões e agregações a cada execução de micro-lote, os Data Marts analíticos são persistidos em modo `overwrite` atômico com `overwriteSchema = true`, mantendo a consistência exata do catálogo e dos KPIs consolidados.

In [ ]:
data_marts_lake = {
    caminhos["gold_dim_produtos"]: df_gold_dim_produtos,
    caminhos["gold_metricas_categorias"]: df_gold_metricas_categorias,
    caminhos["gold_pedidos_rota"]: df_gold_pedidos_rota,
    caminhos["gold_performance_transportadoras"]: df_gold_transportadoras,
    caminhos["gold_distribuicao_geografica"]: df_gold_geografia,
    caminhos["gold_alertas_operacionais"]: df_gold_alertas_operacionais
}

print("Sincronizando Data Marts em Delta Lake (ADLS Gen2) em modo OVERWRITE...")
for path, df_mart in data_marts_lake.items():
    if df_mart.count() == 0:
        print(f"  [AVISO] Data Mart em '{path}' possui 0 registros. Gravação ignorada com segurança.")
        continue
    (df_mart.write
        .format("delta")
        .options(**adls_options)
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact", "true")
        .save(path))
    print(f"  [SUCESSO] Data Mart sincronizado e gravado em: {path}")


## 5. Persistência 2: Replicação Resiliente no Azure SQL Server (`squad2`) (Sincronização / Modo Overwrite)

Sincronização atômica (`TRUNCATE + INSERT` / `mode('overwrite')`) para garantir que as tabelas relacionais reflitam fielmente o estado analítico higienizado, eliminando acúmulo de centenas de milhares de linhas repetidas.

In [ ]:
import pyodbc

# Metadados oficiais das colunas criadas no banco de dados squad2
COLUNAS_SQL_TABELAS = {
    "squad2.gold_dim_produtos": [
        "sku", "nome_produto", "descricao", "id_categoria", "nome_categoria",
        "id_categoria_pai", "nome_categoria_pai", "tipo_categoria", "nome_marca",
        "unidade_medida", "preco_lista", "preco_custo", "margem_bruta_unitaria",
        "margem_lucro_pct", "status_rentabilidade", "is_ativo",
        "status_disponibilidade", "faixa_preco", "gold_processed_at"
    ],
    "squad2.gold_metricas_categorias": [
        "id_categoria", "nome_categoria", "id_categoria_pai", "nome_categoria_pai",
        "tipo_categoria", "is_categoria_raiz", "total_skus_cadastrados", "total_skus_ativos",
        "total_skus_inativos", "taxa_disponibilidade_pct", "preco_medio_lista",
        "preco_minimo_lista", "preco_maximo_lista", "amplitude_preco",
        "total_marcas_distintas", "gold_calculated_at"
    ],
    "squad2.gold_logistica_pedidos_rota": [
        "data_evento", "total_eventos", "total_pedidos_unicos",
        "total_em_separacao", "total_coletados", "total_em_transito", "total_em_rota",
        "pedidos_em_rota_ultimas_2h", "total_entregues", "total_falhas",
        "total_cancelados", "total_outros_status",
        "total_entregues_no_prazo", "total_entregues_com_atraso", "taxa_sla_entrega_pct",
        "taxa_sucesso_pct", "gold_processed_at"
    ],
    "squad2.gold_performance_transportadoras": [
        "data_evento", "id_transportadora", "volume_movimentado", "total_pedidos_atendidos",
        "volume_entregue", "volume_falhas", "taxa_eficiencia_pct", "ranking_volume_dia",
        "gold_processed_at"
    ],
    "squad2.gold_distribuicao_geografica_clientes": [
        "estado", "total_enderecos", "total_clientes_unicos", "total_enderecos_principais",
        "penetracao_clientes_pct", "gold_processed_at"
    ],
    "squad2.gold_alertas_operacionais": [
        "data_alerta", "dominio", "regra_negocio", "chave_entidade",
        "valor_detectado", "severidade", "gold_processed_at"
    ]
}

def salvar_gold_sql(df, nome_tabela, modo="overwrite"):
    if df is None or df.count() == 0:
        print(f"\n[AVISO DE PROTEÇÃO] Data Mart '{nome_tabela}' sem registros para replicar. Persistência ignorada com segurança.")
        return

    # Filtra e alinha estritamente as colunas esperadas pela tabela de destino no SQL Server
    colunas_destino = COLUNAS_SQL_TABELAS.get(nome_tabela)
    if colunas_destino:
        colunas_disponiveis = [c for c in colunas_destino if c in df.columns]
        df_sql = df.select(colunas_disponiveis)
    else:
        df_sql = df

    print(f"\nReplicando {nome_tabela} no Azure SQL Server (modo {modo.upper()} - {df_sql.count()} linhas)...")
    
    # 1. Tentativa conector nativo sqlserver
    try:
        df_sql.write \
            .format("sqlserver") \
            .option("host", sql_host) \
            .option("port", "1433") \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .option("database", sql_database) \
            .option("dbtable", nome_tabela) \
            .mode(modo) \
            .save()
        print(f"  [SUCESSO] {nome_tabela} gravado em modo {modo} via conector sqlserver.")
        return
    except Exception as e_sql:
        print(f"  Aviso (sqlserver): {e_sql}")

    # 2. Fallback direto pyodbc fast_executemany
    print(f"  -> Ativando fallback pyodbc fast_executemany (modo {modo})...")
    try:
        pdf = df_sql.toPandas()
        conn_str = (
            f"DRIVER={{ODBC Driver 18 for SQL Server}};"
            f"SERVER={sql_host},1433;"
            f"DATABASE={sql_database};"
            f"UID={sql_user};"
            f"PWD={sql_password};"
            f"Encrypt=yes;"
            f"TrustServerCertificate=no;"
            f"Connection Timeout=30;"
        )
        with pyodbc.connect(conn_str) as conn:
            cursor = conn.cursor()
            if nome_tabela == "squad2.gold_dim_produtos":
                cursor.execute("""
                    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'gold_dim_produtos' AND schema_id = SCHEMA_ID('squad2'))
                    BEGIN
                        CREATE TABLE squad2.gold_dim_produtos (
                            sku VARCHAR(64) NULL,
                            nome_produto VARCHAR(255) NULL,
                            descricao VARCHAR(500) NULL,
                            id_categoria BIGINT NULL,
                            nome_categoria VARCHAR(100) NULL,
                            id_categoria_pai BIGINT NULL,
                            nome_categoria_pai VARCHAR(100) NULL,
                            tipo_categoria VARCHAR(50) NULL,
                            nome_marca VARCHAR(100) NULL,
                            unidade_medida VARCHAR(20) NULL,
                            preco_lista FLOAT NULL,
                            preco_custo FLOAT NULL,
                            margem_bruta_unitaria FLOAT NULL,
                            margem_lucro_pct FLOAT NULL,
                            status_rentabilidade VARCHAR(50) NULL,
                            is_ativo BIT NULL,
                            status_disponibilidade VARCHAR(50) NULL,
                            faixa_preco VARCHAR(50) NULL,
                            gold_processed_at DATETIME2 NULL
                        )
                    END
                    ELSE
                    BEGIN
                        IF NOT EXISTS (SELECT * FROM sys.columns WHERE object_id = OBJECT_ID('squad2.gold_dim_produtos') AND name = 'preco_custo')
                            ALTER TABLE squad2.gold_dim_produtos ADD preco_custo FLOAT NULL;
                        IF NOT EXISTS (SELECT * FROM sys.columns WHERE object_id = OBJECT_ID('squad2.gold_dim_produtos') AND name = 'margem_bruta_unitaria')
                            ALTER TABLE squad2.gold_dim_produtos ADD margem_bruta_unitaria FLOAT NULL;
                        IF NOT EXISTS (SELECT * FROM sys.columns WHERE object_id = OBJECT_ID('squad2.gold_dim_produtos') AND name = 'margem_lucro_pct')
                            ALTER TABLE squad2.gold_dim_produtos ADD margem_lucro_pct FLOAT NULL;
                        IF NOT EXISTS (SELECT * FROM sys.columns WHERE object_id = OBJECT_ID('squad2.gold_dim_produtos') AND name = 'status_rentabilidade')
                            ALTER TABLE squad2.gold_dim_produtos ADD status_rentabilidade VARCHAR(50) NULL;
                    END
                """)
                conn.commit()
            elif nome_tabela == "squad2.gold_logistica_pedidos_rota":
                cursor.execute("""
                    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'gold_logistica_pedidos_rota' AND schema_id = SCHEMA_ID('squad2'))
                    BEGIN
                        CREATE TABLE squad2.gold_logistica_pedidos_rota (
                            data_evento DATE NULL,
                            total_eventos BIGINT NULL,
                            total_pedidos_unicos BIGINT NULL,
                            total_em_separacao BIGINT NULL,
                            total_coletados BIGINT NULL,
                            total_em_transito BIGINT NULL,
                            total_em_rota BIGINT NULL,
                            pedidos_em_rota_ultimas_2h BIGINT NULL,
                            total_entregues BIGINT NULL,
                            total_falhas BIGINT NULL,
                            total_cancelados BIGINT NULL,
                            total_outros_status BIGINT NULL,
                            total_entregues_no_prazo BIGINT NULL,
                            total_entregues_com_atraso BIGINT NULL,
                            taxa_sla_entrega_pct FLOAT NULL,
                            taxa_sucesso_pct FLOAT NULL,
                            gold_processed_at DATETIME2 NULL
                        )
                    END
                    ELSE
                    BEGIN
                        IF NOT EXISTS (SELECT * FROM sys.columns WHERE object_id = OBJECT_ID('squad2.gold_logistica_pedidos_rota') AND name = 'total_em_separacao')
                            ALTER TABLE squad2.gold_logistica_pedidos_rota ADD total_em_separacao BIGINT NULL;
                        IF NOT EXISTS (SELECT * FROM sys.columns WHERE object_id = OBJECT_ID('squad2.gold_logistica_pedidos_rota') AND name = 'total_cancelados')
                            ALTER TABLE squad2.gold_logistica_pedidos_rota ADD total_cancelados BIGINT NULL;
                        IF NOT EXISTS (SELECT * FROM sys.columns WHERE object_id = OBJECT_ID('squad2.gold_logistica_pedidos_rota') AND name = 'total_outros_status')
                            ALTER TABLE squad2.gold_logistica_pedidos_rota ADD total_outros_status BIGINT NULL;
                    END
                """)
                conn.commit()
            elif nome_tabela == "squad2.gold_alertas_operacionais":
                cursor.execute("""
                    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'gold_alertas_operacionais' AND schema_id = SCHEMA_ID('squad2'))
                    BEGIN
                        CREATE TABLE squad2.gold_alertas_operacionais (
                            data_alerta DATETIME2 NULL,
                            dominio VARCHAR(100) NULL,
                            regra_negocio VARCHAR(200) NULL,
                            chave_entidade VARCHAR(100) NULL,
                            valor_detectado VARCHAR(255) NULL,
                            severidade VARCHAR(50) NULL,
                            gold_processed_at DATETIME2 NULL
                        )
                    END
                """)
                conn.commit()

            # Em modo overwrite, limpa a tabela para sincronização do snapshot atual
            if modo == "overwrite":
                cursor.execute(f"TRUNCATE TABLE {nome_tabela}")
                conn.commit()

            cursor.fast_executemany = True
            cols = [f"[{c}]" for c in pdf.columns]
            placeholders = ", ".join(["?"] * len(pdf.columns))
            sql = f"INSERT INTO {nome_tabela} ({', '.join(cols)}) VALUES ({placeholders})"
            
            pdf_clean = pdf.where(pd.notnull(pdf), None)
            valores = [tuple(x) for x in pdf_clean.to_numpy()]
            cursor.executemany(sql, valores)
            conn.commit()
            print(f"  [SUCESSO] {len(pdf)} registros sincronizados (modo {modo}) via pyodbc em {nome_tabela}.")
    except Exception as e_py:
        raise RuntimeError(f"Falha ao persistir {nome_tabela} no SQL Server: {e_py}")

salvar_gold_sql(df_gold_dim_produtos, "squad2.gold_dim_produtos", modo="overwrite")
salvar_gold_sql(df_gold_metricas_categorias, "squad2.gold_metricas_categorias", modo="overwrite")
salvar_gold_sql(df_gold_pedidos_rota, "squad2.gold_logistica_pedidos_rota", modo="overwrite")
salvar_gold_sql(df_gold_transportadoras, "squad2.gold_performance_transportadoras", modo="overwrite")
salvar_gold_sql(df_gold_geografia, "squad2.gold_distribuicao_geografica_clientes", modo="overwrite")
salvar_gold_sql(df_gold_alertas_operacionais, "squad2.gold_alertas_operacionais", modo="overwrite")


## 6. Auditoria de Fechamento Contábil e Relatório Executivo Gold

In [ ]:
print("=" * 80)
print("RELATÓRIO EXECUTIVO DE FECHAMENTO ANALÍTICO - CAMADA GOLD (SQUAD 2)")
print("=" * 80)

# Auditoria Fechamento Contábil Grupo 1
if df_gold_metricas_categorias.count() > 0:
    chk_produtos = df_gold_metricas_categorias.select(
        F.sum("total_skus_cadastrados").alias("cadastrados"),
        F.sum("total_skus_ativos").alias("ativos"),
        F.sum("total_skus_inativos").alias("inativos")
    ).first()

    fechamento_ok = chk_produtos["cadastrados"] == (chk_produtos["ativos"] + chk_produtos["inativos"])

    print(f"Fechamento Contábil de Catálogo:")
    print(f"  Total SKUs Cadastrados: {chk_produtos['cadastrados']}")
    print(f"  Ativos + Inativos:      {chk_produtos['ativos'] + chk_produtos['inativos']}")
    print(f"  Status Fechamento:      {'[100% EXATO - FECHAMENTO CONTÁBIL CONFIRMADO]' if fechamento_ok else '[ERRO DE CONCILIAÇÃO]'}")
else:
    print("Fechamento Contábil de Catálogo: [PULADO - SEM DADOS NA GOLD]")

print(f"\nResumo dos 5 Data Marts Analíticos da Squad 2:")
print(f"  1. gold_dim_produtos:                      {df_gold_dim_produtos.count()} produtos modelados")
if df_gold_dim_produtos.count() > 0 and "preco_custo" in df_gold_dim_produtos.columns:
    skus_com_custo = df_gold_dim_produtos.filter(F.col("preco_custo").isNotNull()).count()
    if skus_com_custo > 0:
        avg_margem = df_gold_dim_produtos.filter(F.col("preco_custo").isNotNull()).select(F.round(F.avg("margem_lucro_pct"), 2)).first()[0]
        print(f"     -> SKUs com Custo Cadastrado (Schema Evolution): {skus_com_custo} | Margem Média: {avg_margem}%")
print(f"  2. gold_metricas_categorias:               {df_gold_metricas_categorias.count()} categorias com métricas agregadas")
print(f"  3. gold_logistica_pedidos_rota:            {df_gold_pedidos_rota.count()} dias operacionais analisados")
print(f"  4. gold_performance_transportadoras:       {df_gold_transportadoras.count()} linhas de ranking de parceiros")
print(f"  5. gold_distribuicao_geografica_clientes:  {df_gold_geografia.count()} UFs com análise de penetração")
print(f"  6. gold_alertas_operacionais:              {df_gold_alertas_operacionais.count()} alertas de negócio ativos")
print("=" * 80)
print("Pipeline da Camada Gold concluído e certificado com Destino Duplo!")